In [10]:
MOT_ROOT_PATH = "/content/drive/MyDrive/Projeto Nível I em Cont e Proc de Sinais I/SoccerTrack v2/mot"
NEW_MOT_ROOT_PATH = "/content/drive/MyDrive/dataset"
N_FRAMES_PER_VIDEO = 36

In [11]:
import cv2
import os
from google.colab import drive
from collections import defaultdict
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
class MOT_Annotation:
    def __init__(self, frame, id, x, y, w, h):
        self.frame = frame
        self.id = id
        self.x = x
        self.y = y
        self.w = w
        self.h = h

    def __str__(self):
        return f"Frame: {self.frame}, ID: {self.id}, X: {self.x}, Y: {self.y}, W: {self.w}, H: {self.h}"
def load_mot_annotations(root_path: str, match_id: int):
    annotation_file = f"{root_path}/{match_id}.txt"

    annotations_by_frame = defaultdict(list)

    with open(annotation_file, "r") as file:
        for line in file:
            if not line.strip():
                continue

            properties = line.strip().split(",")

            frame = int(properties[0])
            track_id = int(properties[1])
            x = float(properties[2])
            y = float(properties[3])
            w = float(properties[4])
            h = float(properties[5])

            annotation = MOT_Annotation(
                frame,
                track_id,
                x,
                y,
                w,
                h
            )

            annotations_by_frame[frame].append(annotation)

    return annotations_by_frame

In [13]:
clips = os.listdir(f"{MOT_ROOT_PATH}/clips")

for clip in clips:

    if not clip.endswith(".mp4"):
        continue

    match_id = clip.split(".")[0]
    clip_full_path = f"{MOT_ROOT_PATH}/clips/{clip}"

    print(f"Loading {match_id} at {clip_full_path}")

    cap = cv2.VideoCapture(clip_full_path)

    if not cap.isOpened():
        print("Error: Could not open video file.")
        continue

    print("Video file opened successfully!")

    # Load annotations for this match
    annotations = load_mot_annotations(
        MOT_ROOT_PATH,
        match_id
    )

    # --------------------------------------------------
    # Create output directories
    # --------------------------------------------------

    image_output_path = (
        f"{NEW_MOT_ROOT_PATH}/images/{match_id}"
    )

    label_output_path = (
        f"{NEW_MOT_ROOT_PATH}/labels/{match_id}"
    )

    os.makedirs(image_output_path, exist_ok=True)
    os.makedirs(label_output_path, exist_ok=True)

    # --------------------------------------------------
    # Video information
    # --------------------------------------------------

    frame_width = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    frame_height = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    frame_index = 0

    while True:

        success, frame = cap.read()

        if not success:
            break

        if frame_index % N_FRAMES_PER_VIDEO == 0:

            # ------------------------------------------
            # Get annotations for this frame
            # ------------------------------------------

            frame_annotations = annotations.get(
                frame_index,
                []
            )

            # If there are no players, skip this frame
            if not frame_annotations:
                frame_index += 1
                continue

            # ------------------------------------------
            # Filename
            # ------------------------------------------

            filename = (
                f"{match_id}_frame_{frame_index:06d}"
            )

            image_filename = f"{filename}.jpeg"
            label_filename = f"{filename}.txt"

            image_path = os.path.join(
                image_output_path,
                image_filename
            )

            label_path = os.path.join(
                label_output_path,
                label_filename
            )

            # ------------------------------------------
            # Save image
            # ------------------------------------------

            cv2.imwrite(
                image_path,
                frame
            )

            # ------------------------------------------
            # Save YOLO annotations
            # ------------------------------------------

            with open(label_path, "w") as file:

                for annotation in frame_annotations:

                    # MOT format:
                    #
                    # x, y = top-left corner
                    # w, h = bounding-box dimensions
                    #
                    # YOLO format:
                    #
                    # x_center
                    # y_center
                    # width
                    # height
                    #
                    # All normalized to [0, 1]

                    x_center = (
                        annotation.x +
                        annotation.w / 2
                    ) / frame_width

                    y_center = (
                        annotation.y +
                        annotation.h / 2
                    ) / frame_height

                    width = (
                        annotation.w /
                        frame_width
                    )

                    height = (
                        annotation.h /
                        frame_height
                    )

                    # Class 0 = player
                    file.write(
                        f"0 "
                        f"{x_center:.6f} "
                        f"{y_center:.6f} "
                        f"{width:.6f} "
                        f"{height:.6f}\n"
                    )

        frame_index += 1

    cap.release()

    print(f"Finished {match_id}")

Loading 118575 at /content/drive/MyDrive/Projeto Nível I em Cont e Proc de Sinais I/SoccerTrack v2/mot/clips/118575.mp4
Video file opened successfully!
Finished 118575
Loading 117092 at /content/drive/MyDrive/Projeto Nível I em Cont e Proc de Sinais I/SoccerTrack v2/mot/clips/117092.mp4
Video file opened successfully!
Finished 117092
Loading 117093 at /content/drive/MyDrive/Projeto Nível I em Cont e Proc de Sinais I/SoccerTrack v2/mot/clips/117093.mp4
Video file opened successfully!
Finished 117093
Loading 118577 at /content/drive/MyDrive/Projeto Nível I em Cont e Proc de Sinais I/SoccerTrack v2/mot/clips/118577.mp4
Video file opened successfully!
Finished 118577
Loading 118576 at /content/drive/MyDrive/Projeto Nível I em Cont e Proc de Sinais I/SoccerTrack v2/mot/clips/118576.mp4
Video file opened successfully!
Finished 118576
Loading 128058 at /content/drive/MyDrive/Projeto Nível I em Cont e Proc de Sinais I/SoccerTrack v2/mot/clips/128058.mp4
Video file opened successfully!
Fi